# Data Inventory EDA

This notebook reads every clean CSV in `data/processed`, checks schema, null coverage, pandas dtypes, duplicate URLs, and basic salary/skill coverage.

TopDev is kept in the full inventory and demand exploration. It is excluded only from numeric salary analysis when no `salary_min` or `salary_max` exists.

In [12]:
from pathlib import Path
import ast
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook convenience fallback
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/processed")


REPO_ROOT = find_repo_root()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
REPORT_DIR = REPO_ROOT / "data" / "reports"

print(f"Repo root: {REPO_ROOT}")
print(f"Processed data dir: {PROCESSED_DIR}")

Repo root: G:\project\Vietnam IT Job Market Intelligence
Processed data dir: G:\project\Vietnam IT Job Market Intelligence\data\processed


## Load all clean CSVs

The load step keeps `_input_file` so sample/test/full/salary runs can be inspected separately before deduping.

In [13]:
clean_paths = sorted(PROCESSED_DIR.glob("*_clean.csv"))
if not clean_paths:
    raise FileNotFoundError(f"No *_clean.csv files found in {PROCESSED_DIR}")

frames_by_file = {}
for path in clean_paths:
    frame = pd.read_csv(path, encoding="utf-8-sig")
    frame["_input_file"] = path.name
    frames_by_file[path.name] = frame

jobs = pd.concat(frames_by_file.values(), ignore_index=True, sort=False)
base_columns = [column for column in next(iter(frames_by_file.values())).columns if column != "_input_file"]

numeric_columns = ["salary_min", "salary_max", "experience_min", "experience_max"]
for column in numeric_columns:
    if column in jobs.columns:
        jobs[column] = pd.to_numeric(jobs[column], errors="coerce")

date_columns = ["scraped_at", "posted_at", "valid_through"]
for column in date_columns:
    if column in jobs.columns:
        jobs[f"{column}_parsed"] = pd.to_datetime(jobs[column], errors="coerce", utc=True)

jobs["_has_numeric_salary"] = jobs[["salary_min", "salary_max"]].notna().any(axis=1)
jobs["_salary_midpoint"] = jobs[["salary_min", "salary_max"]].mean(axis=1)
jobs["_has_skills"] = jobs["skills"].fillna("").astype(str).str.strip().ne("")
jobs["_has_experience"] = jobs["experience_raw"].fillna("").astype(str).str.strip().ne("")
jobs["_has_description"] = jobs["description"].fillna("").astype(str).str.strip().ne("")
jobs["_quality_score"] = (
    jobs["_has_numeric_salary"].astype(int) * 100
    + jobs["_has_skills"].astype(int) * 10
    + jobs["_has_experience"].astype(int) * 5
    + jobs["_has_description"].astype(int)
)

print(f"Clean CSV files loaded: {len(clean_paths)}")
print(f"Rows loaded: {len(jobs):,}")
print(f"Unique URLs: {jobs['url'].nunique():,}")
display(pd.DataFrame({"input_file": [path.name for path in clean_paths]}))

Clean CSV files loaded: 8
Rows loaded: 1,662
Unique URLs: 1,433


,input_file
0,itviec_20260701_100_clean.csv
1,itviec_salary_20260702_clean.csv
2,itviec_sample_50_clean.csv
3,itviec_test_2_clean.csv
4,topcv_20260702_100_clean.csv
5,topcv_20260702_test_5_clean.csv
6,topcv_salary_20260702_clean.csv
7,topdev_20260701_100_clean.csv


## Inventory by file and source

In [14]:
def non_empty_count(series):
    return int(series.fillna("").astype(str).str.strip().ne("").sum())


inventory = (
    jobs.groupby(["_input_file", "source"], dropna=False)
    .agg(
        rows=("url", "size"),
        unique_urls=("url", "nunique"),
        parse_ok_rows=("parse_status", lambda s: int(s.astype(str).eq("ok").sum())),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        title_non_empty=("title", non_empty_count),
        company_non_empty=("company", non_empty_count),
        location_non_empty=("location", non_empty_count),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
        description_non_empty=("description", non_empty_count),
    )
    .reset_index()
)
inventory["parse_ok_rate"] = inventory["parse_ok_rows"] / inventory["rows"]
inventory["salary_numeric_fill_rate"] = inventory["numeric_salary_rows"] / inventory["rows"]
inventory["skills_fill_rate"] = inventory["skills_non_empty"] / inventory["rows"]
inventory["experience_fill_rate"] = inventory["experience_non_empty"] / inventory["rows"]

display(inventory.sort_values(["source", "_input_file"]).reset_index(drop=True))

source_inventory = (
    jobs.groupby("source", dropna=False)
    .agg(
        rows=("url", "size"),
        unique_urls=("url", "nunique"),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        parse_ok_rows=("parse_status", lambda s: int(s.astype(str).eq("ok").sum())),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
    )
    .reset_index()
)
source_inventory["salary_numeric_fill_rate"] = source_inventory["numeric_salary_rows"] / source_inventory["rows"]
source_inventory["parse_ok_rate"] = source_inventory["parse_ok_rows"] / source_inventory["rows"]
display(source_inventory.sort_values("source"))

,_input_file,source,rows,unique_urls,parse_ok_rows,numeric_salary_rows,title_non_empty,company_non_empty,location_non_empty,salary_raw_non_empty,skills_non_empty,experience_non_empty,description_non_empty,parse_ok_rate,salary_numeric_fill_rate,skills_fill_rate,experience_fill_rate
0,itviec_20260701_100_clean.csv,itviec,100,100,100,27,100,100,100,100,100,82,100,1.0,0.270000,1.000000,0.820000
1,itviec_salary_20260702_clean.csv,itviec,750,750,750,207,750,750,750,750,750,658,750,1.0,0.276000,1.000000,0.877333
2,itviec_sample_50_clean.csv,itviec,30,30,30,8,30,30,30,30,30,25,30,1.0,0.266667,1.000000,0.833333
3,itviec_test_2_clean.csv,itviec,2,2,2,0,2,2,2,2,2,1,2,1.0,0.000000,1.000000,0.500000
4,topcv_20260702_100_clean.csv,topcv,100,100,100,55,100,100,100,95,77,81,100,1.0,0.550000,0.770000,0.810000
5,topcv_20260702_test_5_clean.csv,topcv,5,5,5,2,5,5,5,5,4,4,5,1.0,0.400000,0.800000,0.800000
6,topcv_salary_20260702_clean.csv,topcv,575,575,575,298,575,575,575,526,446,488,575,1.0,0.518261,0.775652,0.848696
7,topdev_20260701_100_clean.csv,topdev,100,100,100,0,100,100,100,100,100,83,100,1.0,0.000000,1.000000,0.830000


,source,rows,unique_urls,numeric_salary_rows,parse_ok_rows,salary_raw_non_empty,skills_non_empty,experience_non_empty,salary_numeric_fill_rate,parse_ok_rate
0,itviec,882,750,242,882,882,882,766,0.274376,1.0
1,topcv,680,583,355,680,626,527,573,0.522059,1.0
2,topdev,100,100,0,100,100,100,83,0.000000,1.0


## Schema, dtypes, and parsed dates

Raw CSV date columns read as `object`; the notebook adds parsed UTC datetime columns for EDA without changing source files.

In [15]:
schema_rows = []
for file_name, frame in frames_by_file.items():
    for column in frame.columns:
        schema_rows.append({"_input_file": file_name, "column": column, "dtype": str(frame[column].dtype)})

schema_long = pd.DataFrame(schema_rows)
schema_presence = (
    schema_long.assign(present=True)
    .pivot_table(index="column", columns="_input_file", values="present", aggfunc="any", fill_value=False)
    .reset_index()
)
display(schema_presence)

dtype_matrix = schema_long.pivot(index="column", columns="_input_file", values="dtype").reset_index()
display(dtype_matrix)

dtype_checks = []
for column in numeric_columns:
    dtype_checks.append(
        {
            "column": column,
            "expected_for_eda": "numeric",
            "actual_dtype_after_load": str(jobs[column].dtype),
            "ok": pd.api.types.is_numeric_dtype(jobs[column]),
        }
    )
for column in date_columns:
    parsed = f"{column}_parsed"
    dtype_checks.append(
        {
            "column": parsed,
            "expected_for_eda": "datetime64[ns, UTC]",
            "actual_dtype_after_load": str(jobs[parsed].dtype),
            "ok": pd.api.types.is_datetime64_any_dtype(jobs[parsed]),
        }
    )
display(pd.DataFrame(dtype_checks))

date_parse_summary = []
for column in date_columns:
    parsed = f"{column}_parsed"
    date_parse_summary.append(
        {
            "column": column,
            "raw_non_empty": non_empty_count(jobs[column]),
            "parsed_non_null": int(jobs[parsed].notna().sum()),
            "parse_null_rate": round(float(jobs[parsed].isna().mean()), 4),
        }
    )
display(pd.DataFrame(date_parse_summary))

_input_file,column,itviec_20260701_100_clean.csv,itviec_salary_20260702_clean.csv,itviec_sample_50_clean.csv,itviec_test_2_clean.csv,topcv_20260702_100_clean.csv,topcv_20260702_test_5_clean.csv,topcv_salary_20260702_clean.csv,topdev_20260701_100_clean.csv
0,_input_file,True,True,True,True,True,True,True,True
1,company,True,True,True,True,True,True,True,True
2,description,True,True,True,True,True,True,True,True
3,employment_type,True,True,True,True,True,True,True,True
4,experience_max,True,True,True,True,True,True,True,True
5,experience_min,True,True,True,True,True,True,True,True
6,experience_raw,True,True,True,True,True,True,True,True
7,job_id,True,True,True,True,True,True,True,True
8,location,True,True,True,True,True,True,True,True
9,location_cities,True,True,True,True,True,True,True,True


_input_file,column,itviec_20260701_100_clean.csv,itviec_salary_20260702_clean.csv,itviec_sample_50_clean.csv,itviec_test_2_clean.csv,topcv_20260702_100_clean.csv,topcv_20260702_test_5_clean.csv,topcv_salary_20260702_clean.csv,topdev_20260701_100_clean.csv
0,_input_file,object,object,object,object,object,object,object,object
1,company,object,object,object,object,object,object,object,object
2,description,object,object,object,object,object,object,object,object
3,employment_type,object,object,object,object,object,object,object,object
4,experience_max,float64,float64,float64,float64,float64,float64,float64,float64
5,experience_min,float64,float64,float64,float64,float64,float64,float64,float64
6,experience_raw,object,object,object,object,object,object,object,object
7,job_id,object,object,object,object,object,object,object,object
8,location,object,object,object,object,object,object,object,object
9,location_cities,object,object,object,object,object,object,object,object


,column,expected_for_eda,actual_dtype_after_load,ok
0,salary_min,numeric,float64,True
1,salary_max,numeric,float64,True
2,experience_min,numeric,float64,True
3,experience_max,numeric,float64,True
4,scraped_at_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True
5,posted_at_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True
6,valid_through_parsed,"datetime64[ns, UTC]","datetime64[ns, UTC]",True


,column,raw_non_empty,parsed_non_null,parse_null_rate
0,scraped_at,1662,1662,0.0000
1,posted_at,1662,1662,0.0000
2,valid_through,1662,982,0.4091


## Null and fill-rate checks

In [16]:
def null_summary(frame, columns):
    rows = []
    for column in columns:
        rows.append(
            {
                "column": column,
                "dtype": str(frame[column].dtype),
                "non_null": int(frame[column].notna().sum()),
                "non_empty": non_empty_count(frame[column]),
                "nulls": int(frame[column].isna().sum()),
                "null_rate": round(float(frame[column].isna().mean()), 4),
            }
        )
    return pd.DataFrame(rows)


overall_nulls = null_summary(jobs, [*base_columns, "_input_file"])
display(overall_nulls)

source_nulls = []
for source, group in jobs.groupby("source", dropna=False):
    summary = null_summary(group, base_columns)
    summary.insert(0, "source", source)
    source_nulls.append(summary)
source_nulls = pd.concat(source_nulls, ignore_index=True)
display(source_nulls.sort_values(["source", "null_rate", "column"], ascending=[True, False, True]).head(80))

,column,dtype,non_null,non_empty,nulls,null_rate
0,source,object,1662,1662,0,0.0000
1,url,object,1662,1662,0,0.0000
2,job_id,object,1662,1662,0,0.0000
3,title,object,1662,1662,0,0.0000
4,company,object,1662,1662,0,0.0000
5,location,object,1662,1662,0,0.0000
6,salary_raw,object,1608,1608,54,0.0325
7,salary_min,float64,569,569,1093,0.6576
8,salary_max,float64,584,584,1078,0.6486
9,salary_currency,object,592,592,1070,0.6438


,source,column,dtype,non_null,non_empty,nulls,null_rate
13,itviec,experience_max,float64,184,184,698,0.7914
7,itviec,salary_min,float64,216,216,666,0.7551
8,itviec,salary_max,float64,238,238,644,0.7302
9,itviec,salary_currency,object,240,240,642,0.7279
12,itviec,experience_min,float64,766,766,116,0.1315
...,...,...,...,...,...,...,...
60,topdev,skills,object,100,100,0,0.0000
50,topdev,source,object,100,100,0,0.0000
53,topdev,title,object,100,100,0,0.0000
51,topdev,url,object,100,100,0,0.0000


## Duplicate URL handling

EDA keeps both all-row and deduped-by-URL views. The deduped view keeps the row with the richest parsed fields for each URL.

In [17]:
duplicate_mask = jobs["url"].duplicated(keep=False)
duplicate_rows = jobs.loc[duplicate_mask].sort_values(["url", "_quality_score", "_input_file"])

print(f"Rows in duplicate URL groups: {len(duplicate_rows):,}")
print(f"Unique duplicated URLs: {duplicate_rows['url'].nunique():,}")
display(duplicate_rows[["source", "_input_file", "url", "title", "company", "_quality_score"]].head(50))

jobs_deduped = (
    jobs.sort_values(["url", "_quality_score", "_input_file"])
    .drop_duplicates("url", keep="last")
    .copy()
)

deduped_summary = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(
        rows=("url", "size"),
        numeric_salary_rows=("_has_numeric_salary", "sum"),
        salary_raw_non_empty=("salary_raw", non_empty_count),
        skills_non_empty=("skills", non_empty_count),
        experience_non_empty=("experience_raw", non_empty_count),
    )
    .reset_index()
)
deduped_summary["salary_numeric_fill_rate"] = deduped_summary["numeric_salary_rows"] / deduped_summary["rows"]

print(f"Rows after URL dedupe: {len(jobs_deduped):,}")
display(deduped_summary.sort_values("source"))

Rows in duplicate URL groups: 421
Unique duplicated URLs: 192


,source,_input_file,url,title,company,_quality_score
12,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
112,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
863,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-mb-bank-1315,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents),MB Bank,16
49,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16
149,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-ml-python-typesript-net-adk-truong-minh-thinh-technology-joint-stock-co...,"AI Agent Engineer (ML, Python, TypeSript, .NET, ADK)",TRUONG MINH THINH TECHNOLOGY JOINT STOCK COMPANY,16
28,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116
128,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-agent-engineer-python-llm-kds-vietnam-5513,"AI Agent Engineer (Python, LLM)",KDS Vietnam,116
16,itviec,itviec_20260701_100_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16
116,itviec,itviec_salary_20260702_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16
869,itviec,itviec_sample_50_clean.csv,https://itviec.com/it-jobs/ai-automation-engineer-javascript-python-llm-bloomentum-2901,"AI Automation Engineer (JavaScript, Python, LLM)",Bloomentum,16


Rows after URL dedupe: 1,433


,source,rows,numeric_salary_rows,salary_raw_non_empty,skills_non_empty,experience_non_empty,salary_numeric_fill_rate
0,itviec,750,207,750,750,658,0.276000
1,topcv,583,305,534,454,495,0.523156
2,topdev,100,0,100,100,83,0.000000


## TopDev coverage

TopDev is useful for inventory, demand, skills, location, and experience checks. It should not contribute to salary numeric summaries unless future crawls expose numeric salaries.

In [18]:
topdev = jobs_deduped[jobs_deduped["source"].eq("topdev")].copy()
print(f"TopDev deduped rows: {len(topdev):,}")
print(f"TopDev numeric salary rows: {int(topdev['_has_numeric_salary'].sum()):,}")

display(topdev["salary_raw"].fillna("<NA>").astype(str).str.strip().replace("", "<empty>").value_counts().rename_axis("salary_raw").reset_index(name="rows"))
display(topdev[["title", "company", "location", "salary_raw", "experience_raw", "skills", "url"]].head(20))

TopDev deduped rows: 100
TopDev numeric salary rows: 0


,salary_raw,rows
0,Login to view salary,100


,title,company,location,salary_raw,experience_raw,skills,url
1649,"[2026_SVXS_CNTT-DL]_Sinh viên tài năng Khối Công nghệ thông tin, Dữ liệu",Vietcombank,Hà Nội,Login to view salary,fresher,"Computer Science, Data Engineer, AI",https://topdev.vn/detail-jobs/2026-svxs-cntt-dl-sinh-vien-tai-nang-khoi-cong-nghe-thong-tin-du-lieu-vietcombank-2111848
1573,Agentic Engineer (Python/Go/C/C++ Backend & AI Agents) - Khối Công nghệ thông tin (2026TD450887),MBBANK,Hà Nội,Login to view salary,fresher,"Python, Go, C/C++, AI, PostgreSQL, Tensorflow, Computer Vision, Docker, Kubernetes, MongoDB",https://topdev.vn/detail-jobs/agentic-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-2026td450887...
1575,AI Agent Engineer (Middle/Senior),"CyberLogitec Vietnam Co., Ltd.",Hồ Chí Minh,Login to view salary,5+ years,"Python, REST API, AI, React, FastAPI, Vue, PostgreSQL, Redis",https://topdev.vn/detail-jobs/ai-agent-engineer-middle-senior-cyberlogitec-vietnam-co-ltd-2115298
1567,AI & DATA ENGINEER,CÔNG TY TNHH DREAM TALENT,Hồ Chí Minh,Login to view salary,Minimum 3 years,"SQL, Python, Data Engineer, AWS, Azure, GCP",https://topdev.vn/detail-jobs/ai-data-engineer-cong-ty-tnhh-dream-talent-2115438
1578,AI Engineer,AETOSKY PTE. LTD,Hà Nội,Login to view salary,4–8 years,"Python, Git, Docker, Database, Machine Learning, API, AI",https://topdev.vn/detail-jobs/ai-engineer-aetosky-pte-ltd-2115966
1591,AI Engineer (Computer Vision/ NLP/ LLM) - Khối Công Nghệ Thông Tin (2026TD450582),MBBANK,Hà Nội,Login to view salary,NaN,"Python, Machine Learning, C/C++, AI, Deep Learning, Tensorflow, FastAPI, Docker",https://topdev.vn/detail-jobs/ai-engineer-computer-vision-nlp-llm-khoi-cong-nghe-thong-tin-2026td450582-mbbank-2113056
1565,AI ENGINEER,CÔNG TY TNHH PHẦN MỀM NAM LONG,Hồ Chí Minh,Login to view salary,2+ năm,"Python, API, AI",https://topdev.vn/detail-jobs/ai-engineer-cong-ty-tnhh-phan-mem-nam-long-2114873
1571,AI FULLSTACK DEVELOPER,CÔNG TY TNHH SOCOTEC VIỆT NAM,Hà Nội,Login to view salary,NaN,"Python, AI, React",https://topdev.vn/detail-jobs/ai-fullstack-developer-cong-ty-tnhh-socotec-viet-nam-2110330
1574,AI Software Engineer (Python/Go/C/C++ Backend & AI Agents) - Khối Công nghệ thông tin (2026TD450888),MBBANK,Hà Nội,Login to view salary,fresher,"Python, Go, C/C++, AI, PostgreSQL, Tensorflow, Computer Vision, Docker, Kubernetes, MongoDB",https://topdev.vn/detail-jobs/ai-software-engineer-python-go-c-c-backend-ai-agents-khoi-cong-nghe-thong-tin-2026td45...
1612,Backend Developer (Java),NGÂN HÀNG THƯƠNG MẠI CỔ PHẦN SÀI GÒN TÀI LỘC (SACOMBANK),Hồ Chí Minh,Login to view salary,NaN,"OOP, Oracle, Java, Restful Api, Spring Boot, Docker, Kubernetes, Redis, DevOps, iOS, Android",https://topdev.vn/detail-jobs/backend-developer-java-ngan-hang-thuong-mai-co-phan-sai-gon-tai-loc-sacombank-2110332


## Salary EDA

Salary summaries use the deduped view and only rows with at least one numeric salary bound. Currencies and periods are not mixed.

In [19]:
salary_rows = jobs_deduped[jobs_deduped["_has_numeric_salary"]].copy()


def salary_suspicious_reason(row):
    reasons = []
    if pd.notna(row.get("salary_min")) and row["salary_min"] <= 0:
        reasons.append("salary_min_non_positive")
    if pd.notna(row.get("salary_min")) and pd.notna(row.get("salary_max")) and row["salary_min"] > row["salary_max"]:
        reasons.append("salary_min_gt_max")
    if pd.isna(row.get("salary_currency")) or str(row.get("salary_currency", "")).strip() == "":
        reasons.append("numeric_salary_missing_currency")
    if str(row.get("salary_currency", "")).upper() == "VND" and pd.notna(row.get("_salary_midpoint")) and row["_salary_midpoint"] > 500_000_000:
        reasons.append("vnd_midpoint_gt_500m")
    return ";".join(reasons)


salary_rows["salary_suspicious_reason"] = salary_rows.apply(salary_suspicious_reason, axis=1)
salary_rows_clean = salary_rows[salary_rows["salary_suspicious_reason"].eq("")].copy()
suspicious_salary = salary_rows[salary_rows["salary_suspicious_reason"].ne("")].copy()

salary_coverage = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(rows=("url", "size"), numeric_salary_rows=("_has_numeric_salary", "sum"))
    .reset_index()
)
salary_coverage["numeric_salary_rate"] = salary_coverage["numeric_salary_rows"] / salary_coverage["rows"]
display(salary_coverage.sort_values("source"))

salary_summary = (
    salary_rows_clean.groupby(["source", "salary_currency", "salary_period"], dropna=False)
    .agg(
        salary_count=("url", "nunique"),
        midpoint_median=("_salary_midpoint", "median"),
        midpoint_min=("_salary_midpoint", "min"),
        midpoint_max=("_salary_midpoint", "max"),
        salary_min_median=("salary_min", "median"),
        salary_max_median=("salary_max", "median"),
    )
    .reset_index()
    .sort_values(["source", "salary_currency", "salary_period"])
)
display(salary_summary)

print(f"Suspicious salary rows: {len(suspicious_salary):,}")
display(suspicious_salary[["salary_suspicious_reason", "source", "title", "company", "location", "salary_raw", "salary_min", "salary_max", "salary_currency", "salary_period", "url"]])

,source,rows,numeric_salary_rows,numeric_salary_rate
0,itviec,750,207,0.276000
1,topcv,583,305,0.523156
2,topdev,100,0,0.000000


,source,salary_currency,salary_period,salary_count,midpoint_median,midpoint_min,midpoint_max,salary_min_median,salary_max_median
0,itviec,USD,month,133,1500.0,230.0,7000.0,1000.0,2000.0
1,itviec,USD,year,43,1500.0,300.0,3250.0,1000.0,2000.0
2,itviec,VND,month,24,34250000.0,16500000.0,95000000.0,30000000.0,35000000.0
3,itviec,VND,year,2,33750000.0,17500000.0,50000000.0,15000000.0,35000000.0
4,topcv,USD,month,6,1075.0,600.0,1500.0,650.0,1500.0
5,topcv,VND,month,240,17500000.0,1750000.0,83500000.0,14000000.0,23000000.0
6,topcv,VND,year,56,18750000.0,2500000.0,57500000.0,14500000.0,25000000.0


Suspicious salary rows: 8


,salary_suspicious_reason,source,title,company,location,salary_raw,salary_min,salary_max,salary_currency,salary_period,url
872,salary_min_non_positive,itviec,C++ Developer (JP N3+),FPT Software,Hồ Chí Minh,0 - 500 USD,0.0,500.0,USD,year,https://itviec.com/it-jobs/c-developer-jp-n3-fpt-software-0354
413,salary_min_non_positive,itviec,"Frontend Developer Intern (HTML, CSS, JavaScript)",EZ Games,Hồ Chí Minh,0 - 500 USD,0.0,500.0,USD,month,https://itviec.com/it-jobs/frontend-developer-intern-html-css-javascript-ez-games-0012
609,salary_min_non_positive,itviec,QA/ Manual Tester Intern (QA QC),SRT GROUP,Hồ Chí Minh,"0 - 1,000 USD",0.0,1000.0,USD,year,https://itviec.com/it-jobs/qa-manual-tester-intern-qa-qc-srt-group-2633
227,numeric_salary_missing_currency,itviec,Senior/Expert Data Analyst (5YOE - Banking Finance),F88,Hà Nội,"30,000,000-45,000,000",30000000.0,45000000.0,NaN,year,https://itviec.com/it-jobs/senior-expert-data-analyst-5yoe-banking-finance-f88-2837
452,numeric_salary_missing_currency,itviec,"Sr .NET Backend Developer (ASP.NET, C#, SQL, ReactJS)",GrapeCity,Hà Nội,"30,000,000 - 50,000,000đ",30000000.0,50000000.0,NaN,month,https://itviec.com/it-jobs/sr-net-backend-developer-asp-net-c-sql-reactjs-grapecity-3733
967,numeric_salary_missing_currency,topcv,Fullstack Developer (From 3 YoE) - Domain E-Commerce Product,XIPAT FLEXIBLE SOLUTIONS COMPANY LIMITED,Ha Noi,From 3,3.0,NaN,NaN,year,https://www.topcv.vn/viec-lam/fullstack-developer-from-3-yoe-domain-e-commerce-product/2204957.html
1529,vnd_midpoint_gt_500m,topcv,Kỹ Sư Phần Mềm Hệ Thống Điều Khiển (C#),CÔNG TY TNHH VTI EDUCATION,Nhật Bản,860000000 - 870000000 VND,860000000.0,870000000.0,VND,year,https://www.topcv.vn/viec-lam/ky-su-phan-mem-he-thong-dieu-khien-c/2000152.html
1067,numeric_salary_missing_currency,topcv,Senior Business Analyst (Domain Retail) - Offer Upto 1500$,Công ty Cổ phần MISA,Ha Noi,Upto 1500,NaN,1500.0,NaN,month,https://www.topcv.vn/viec-lam/senior-business-analyst-domain-retail-offer-upto-1500/2220935.html


## Skills and demand exploration

Demand uses all deduped sources, including TopDev. Salary by skill uses clean numeric salary rows only.

In [20]:
def parse_listish(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


skill_rows = jobs_deduped.assign(skill=jobs_deduped["skills"].apply(parse_listish)).explode("skill")
skill_rows = skill_rows[skill_rows["skill"].fillna("").astype(str).str.strip().ne("")].copy()

skill_coverage = (
    jobs_deduped.groupby("source", dropna=False)
    .agg(rows=("url", "size"), rows_with_skills=("_has_skills", "sum"))
    .reset_index()
)
skill_coverage["skills_fill_rate"] = skill_coverage["rows_with_skills"] / skill_coverage["rows"]
display(skill_coverage.sort_values("source"))

demand_by_skill = (
    skill_rows.groupby(["source", "skill"], dropna=False)
    .agg(job_count=("url", "nunique"))
    .reset_index()
    .sort_values(["job_count", "source", "skill"], ascending=[False, True, True])
)
display(demand_by_skill.head(40))

salary_skill_rows = salary_rows_clean.assign(skill=salary_rows_clean["skills"].apply(parse_listish)).explode("skill")
salary_skill_rows = salary_skill_rows[salary_skill_rows["skill"].fillna("").astype(str).str.strip().ne("")].copy()
salary_by_skill = (
    salary_skill_rows.groupby(["source", "skill", "salary_currency", "salary_period"], dropna=False)
    .agg(
        salary_count=("url", "nunique"),
        midpoint_median=("_salary_midpoint", "median"),
        midpoint_min=("_salary_midpoint", "min"),
        midpoint_max=("_salary_midpoint", "max"),
    )
    .reset_index()
)
display(salary_by_skill[salary_by_skill["salary_count"].ge(2)].sort_values(["salary_count", "source", "skill"], ascending=[False, True, True]).head(40))

,source,rows,rows_with_skills,skills_fill_rate
0,itviec,750,750,1.000000
1,topcv,583,454,0.778731
2,topdev,100,100,1.000000


,source,skill,job_count
251,itviec,SQL,252
228,itviec,Python,242
10,itviec,AWS,218
88,itviec,Docker,183
152,itviec,Java,180
100,itviec,English,168
5,itviec,AI,159
84,itviec,DevOps,156
29,itviec,Azure,146
163,itviec,Kubernetes,134


,source,skill,salary_currency,salary_period,salary_count,midpoint_median,midpoint_min,midpoint_max
312,itviec,SQL,USD,month,50,1525.0,400.0,3300.0
280,itviec,Python,USD,month,42,1500.0,300.0,3250.0
14,itviec,AWS,USD,month,36,1650.0,650.0,7000.0
1068,topcv,SQL,VND,month,35,20000000.0,4500000.0,55000000.0
6,itviec,AI,USD,month,29,1600.0,230.0,3250.0
103,itviec,Docker,USD,month,29,1500.0,950.0,3300.0
179,itviec,JavaScript,USD,month,27,1500.0,350.0,4000.0
97,itviec,DevOps,USD,month,26,1450.0,350.0,3300.0
116,itviec,English,USD,month,23,1650.0,900.0,3500.0
176,itviec,Java,USD,month,23,1500.0,350.0,3500.0


## Current-data checks

These checks document the current repository state. If new processed files are added later, rerun the notebook and update the expectations intentionally.

In [21]:
current_checks = {
    "clean_csv_files": len(clean_paths),
    "all_rows_loaded": len(jobs),
    "unique_urls_all_rows": jobs["url"].nunique(),
    "deduped_rows": len(jobs_deduped),
    "parse_status_values": sorted(jobs["parse_status"].dropna().astype(str).unique().tolist()),
    "topdev_numeric_salary_rows": int(jobs_deduped.loc[jobs_deduped["source"].eq("topdev"), "_has_numeric_salary"].sum()),
    "numeric_salary_rows_deduped": int(jobs_deduped["_has_numeric_salary"].sum()),
    "suspicious_salary_rows": len(suspicious_salary),
}
display(pd.DataFrame([current_checks]).T.rename(columns={0: "value"}))

,value
clean_csv_files,8
all_rows_loaded,1662
unique_urls_all_rows,1433
deduped_rows,1433
parse_status_values,[ok]
topdev_numeric_salary_rows,0
numeric_salary_rows_deduped,512
suspicious_salary_rows,8
